In [9]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [10]:
from src.loader import load_league_season
nba_25_26 = load_league_season("nba", "25-26")



Loaded: advanced.csv
Shape: (734, 30)
Loaded: playbyplay.csv
Shape: (734, 26)
Loaded: perposs.csv
Shape: (734, 34)
Loaded: shooting.csv
Shape: (584, 32)


In [11]:
from src.cleaner import clean_stat_table

advanced = clean_stat_table(nba_25_26["advanced"])
perposs = clean_stat_table(nba_25_26["perposs"])
shooting = clean_stat_table(nba_25_26["shooting"])
playbyplay = clean_stat_table(nba_25_26["playbyplay"])

In [12]:
from src.cleaner import clean_stat_table

advanced = clean_stat_table(nba_25_26["advanced"])
perposs = clean_stat_table(nba_25_26["perposs"])
shooting = clean_stat_table(nba_25_26["shooting"])
playbyplay = clean_stat_table(nba_25_26["playbyplay"])

In [13]:
from src.profiles import build_player_profiles

profiles = build_player_profiles(
    advanced=advanced,
    perposs=perposs,
    shooting=shooting,
    playbyplay=playbyplay
)

profiles.shape

(734, 98)

In [14]:
from src.profile_cleanup import clean_profiles

profiles_clean = clean_profiles(profiles)

profiles_clean.shape

(734, 94)

In [15]:
from models.model import build_model_dataset
from src.database import save_df_to_sqlite
from src.profile_cleanup import clean_profiles

model_df = build_model_dataset(
    profiles_clean,
    min_minutes=500
)

model_df.shape
model_df.columns.tolist()

['player',
 'tm',
 'pos',
 'age',
 'g',
 'gs',
 'mp',
 'player_id',
 'pts',
 'ts_pct',
 'efg_pct',
 'fg_pct',
 '2p_pct',
 '3p_pct',
 'ft_pct',
 'three_point_attempt_rate',
 'free_throw_rate',
 'dist',
 '0_3',
 '3_10',
 '10_16',
 '16_3p',
 '0_3_2',
 '3_10_2',
 '10_16_2',
 '16_3p_2',
 'ast',
 'assist_pct',
 'tov',
 'turnover_pct',
 'obpm',
 'ows',
 'ortg',
 'orb',
 'drb',
 'trb',
 'off_reb_pct',
 'def_reb_pct',
 'total_reb_pct',
 'stl',
 'blk',
 'steal_pct',
 'block_pct',
 'pf',
 'drtg',
 'dbpm',
 'dws',
 'per',
 'ws',
 'ws_per_48',
 'bpm',
 'vorp',
 'on_off',
 'pg_pct',
 'sg_pct',
 'sf_pct',
 'pf_pct',
 'c_pct']

In [16]:
profiles_clean.to_csv("../DATA/processed/nba_profiles_master.csv", index=False)
model_df.to_csv("../DATA/processed/nba_25_26_model_dataset_v2.csv", index=False)

save_df_to_sqlite(
    model_df,
    "../DATA/database/parallel_hoops_v2.db",
    "nba_model_dataset_25_26_v2"
)

Saved 435 rows to ../DATA/database/parallel_hoops_v2.db table: nba_model_dataset_25_26_v2


In [ ]:
from importlib import reload
import src.similarity_engine
reload(src.similarity_engine)

from src.similarity_engine import find_similar_players, similarity_feature_report

similarity_feature_report(model_df)

In [24]:
find_similar_players(
    "Nikola Jokić",
    model_df,
    top_n=10
)

,player,tm,pos,age,g,mp,similarity_score
0,Luka Dončić,LAL,PG,26.0,64.0,2289.0,0.912169
1,Shai Gilgeous-Alexander,OKC,PG,27.0,68.0,2259.0,0.887406
2,Cade Cunningham,DET,PG,24.0,64.0,2172.0,0.850578
3,Donovan Mitchell,CLE,SG,29.0,70.0,2342.0,0.842518
4,Jalen Johnson,ATL,SF,24.0,72.0,2532.0,0.841794
5,Deni Avdija,POR,SF,25.0,66.0,2199.0,0.812578
6,Kawhi Leonard,LAC,SF,34.0,65.0,2085.0,0.809591
7,LeBron James,LAL,SF,41.0,60.0,1989.0,0.806308
8,Alperen Şengün,HOU,C,23.0,72.0,2398.0,0.805852
9,Tyrese Maxey,PHI,PG,25.0,70.0,2661.0,0.802811
